In [ ]:
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from accelerate import Accelerator
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from diffusers import DDIMScheduler

In [ ]:
# Dataset for Training Decoder
class PrecomputedCascadeDataset(Dataset):
    """
    Loads the precomputed .pt files produced by the VAE latent precomputation step.
    Each .pt file contains:
        - z_cond: latent [4,64,64]
        - target_img: GT image [3,512,512] in [-1,1]
    """

    def __init__(self, index_jsonl):
        self.items = []
        with open(index_jsonl, "r") as f:
            for line in f:
                path = json.loads(line)["pt"]
                if os.path.exists(path):
                    self.items.append(path)
                else:
                    print(f"[WARN] Missing file, skipped: {path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        pack = torch.load(self.items[i], map_location="cpu")

        return (
            pack["target_img"].float(),       # [-1,1]
            pack["z_cond"].to(torch.float16)  # latent [4,64,64]
        )


In [ ]:
# Multi-scale Prompt Constructing
import math as _m
import torch
import torch.nn as nn
import torch.nn.functional as F

def _g(n, c):
    return _m.gcd(n, c) or 1

class SinCosPos2D(nn.Module):
    def __init__(self, H=64, W=64):
        super().__init__()
        yy, xx = torch.meshgrid(
            torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij'
        )
        pe = torch.stack([
            torch.sin(2*_m.pi*xx), torch.cos(2*_m.pi*xx),
            torch.sin(2*_m.pi*yy), torch.cos(2*_m.pi*yy)
        ], dim=0)  # [4,H,W]
        self.register_buffer("pe", pe.float(), persistent=False)

    def forward(self, B):
        return self.pe.unsqueeze(0).repeat(B, 1, 1, 1)  # [B,4,H,W]

class ResInceptionDilated(nn.Module):
    def __init__(self, ch, mid=None):
        super().__init__()
        mid = mid or ch // 2

        self.pre = nn.Sequential(
            nn.GroupNorm(_g(8, ch), ch),
            nn.SiLU()
        )
        self.reduce = nn.Conv2d(ch, mid, 1)

        self.b1 = nn.Conv2d(mid, mid, 3, padding=1, dilation=1, groups=1, bias=False)
        self.b2 = nn.Conv2d(mid, mid, 3, padding=2, dilation=2, groups=1, bias=False)
        self.b3 = nn.Conv2d(mid, mid, 3, padding=4, dilation=4, groups=1, bias=False)

        self.fuse = nn.Sequential(
            nn.GroupNorm(_g(8, mid*3), mid*3),
            nn.SiLU(),
            nn.Conv2d(mid*3, ch, 1)
        )

    def forward(self, x):
        h = self.pre(x)
        h = self.reduce(h)
        h1 = self.b1(h)
        h2 = self.b2(h)
        h3 = self.b3(h)
        h  = torch.cat([h1, h2, h3], dim=1)
        h  = self.fuse(h)
        return x + h

class UpStage(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
        self.rb   = ResInceptionDilated(ch, mid=ch//2)
    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        x = self.rb(x)
        return x

class LatentAdapter(nn.Module):
    def __init__(self, cz=4, cond_ch=64, width=128,
                 num_blocks_64=3, include_posenc=True):
        super().__init__()
        self.include_posenc = include_posenc

        in_ch = cz + 4

        # 64×64
        self.in_conv = nn.Sequential(
            nn.Conv2d(in_ch, width, 3, padding=1),
            nn.GroupNorm(_g(8, width), width),
            nn.SiLU()
        )
        # 64×64
        self.blocks64 = nn.Sequential(*[ResInceptionDilated(width, mid=width//2) for _ in range(num_blocks_64)])

        self.up1 = UpStage(width)   # -> 128
        self.up2 = UpStage(width)   # -> 256
        self.up3 = UpStage(width)   # -> 512

        def head():
            return nn.Sequential(
                nn.Conv2d(width, cond_ch, 1),
                nn.GroupNorm(_g(8, cond_ch), cond_ch),
                nn.SiLU()
            )
        self.out64  = head()
        self.out128 = head()
        self.out256 = head()
        self.out512 = head()

        self.posenc = SinCosPos2D(64, 64) if include_posenc else None

    def forward(self, z64):
        """
        z64:    [B,4,64,64]
        """
        B, _, H, W = z64.shape
        feats = [z64]


        if self.include_posenc:
            feats.append(self.posenc(B).to(z64.dtype).to(z64.device))

        x = torch.cat(feats, dim=1)             # [B,in_ch,64,64]
        x = self.in_conv(x)                     # [B,width,64,64]
        x = self.blocks64(x)                    # 64×64


        f64  = self.out64(x)                    # [B,cond_ch,64,64]
        x128 = self.up1(x)
        f128 = self.out128(x128)                # [B,cond_ch,128,128]
        x256 = self.up2(x128)
        f256 = self.out256(x256)                # [B,cond_ch,256,256]
        x512 = self.up3(x256)
        f512 = self.out512(x512)                # [B,cond_ch,512,512]

        return {"s64": f64, "s128": f128, "s256": f256, "s512": f512}


In [ ]:
# Unet for decoding
def timestep_embedding(timesteps, dim, max_period=10000):
    """
    Sinusoidal timestep embedding used by Diffusers-style models.
    """
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(
        0, half, dtype=torch.float32, device=timesteps.device
    ) / half)
    args = timesteps.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=1)
    if dim % 2:
        emb = torch.cat([emb, emb[:, :1]], dim=1)
    return emb


class Down(nn.Module):
    """
    Downsampling block:
        Conv → Conv → AvgPool(2)
    Cond channels may be concatenated before the first block.
    """
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)
        self.down = nn.AvgPool2d(2)

    def forward(self, x, t_emb, cond=None):
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        x_down = self.down(x)
        return x, x_down


class Up(nn.Module):
    """
    Upsampling block:
        Interpolate(2×) → concat skip → Conv → Conv
    """
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)

    def forward(self, x, skip, t_emb, cond=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        return x


# 2D ViT Attention (timm)
class TimmAttn2D(nn.Module):
    """
    A lightweight ViT-style attention applied over a 2D spatial map.
    Normalizes input with GroupNorm, flattens to sequences, applies timm Attention,
    then reshapes back to 2D.
    """
    def __init__(self, dim, num_heads=4, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.norm = nn.GroupNorm(_g(8, dim), dim)
        self.attn = ViTAttention(
            dim=dim, num_heads=num_heads,
            qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=proj_drop
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x = self.norm(x)
        x = x.flatten(2).transpose(1, 2)    # [B, HW, C]
        x = self.attn(x)                    # attention over spatial tokens
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return x


class UNet512(nn.Module):
    """
    512×512 UNet architecture used as Stage-2 of the cascade.
    Takes noisy pixel-space images, timestep embeddings, and multi-scale
    conditional features from the latent adapter.
    """

    def __init__(self, base_ch=128, cond_ch=64, time_dim=256):
        super().__init__()

        ch1, ch2, ch3 = base_ch, base_ch * 2, base_ch * 4

        # Timestep MLP
        self.time_mlp = nn.Sequential(
            nn.Linear(320, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # Input conv expects: image + condition (s512)
        self.in_conv = nn.Conv2d(3 + cond_ch, ch1, 3, padding=1)

        # Down path: 512→256→128→64
        self.down1 = Down(ch1, ch1, tdim=time_dim)
        self.down2 = Down(ch1, ch2, tdim=time_dim, cond_ch=cond_ch)
        self.down3 = Down(ch2, ch3, tdim=time_dim, cond_ch=cond_ch)

        # Bottleneck (64×64)
        self.attn64 = TimmAttn2D(dim=ch3, num_heads=4)
        self.mid1 = ResidualBlock2d(ch3 + cond_ch, ch3, time_dim=time_dim)
        self.mid2 = ResidualBlock2d(ch3, ch3, time_dim=time_dim)

        # Up path: 64→128→256→512
        self.up3 = Up(ch3 + ch3, ch2, tdim=time_dim, cond_ch=cond_ch)
        self.up2 = Up(ch2 + ch2, ch1, tdim=time_dim, cond_ch=cond_ch)
        self.up1 = Up(ch1 + ch1, ch1, tdim=time_dim)

        self.out_norm = nn.GroupNorm(8, ch1)
        self.out = nn.Conv2d(ch1, 3, 3, padding=1)

    def forward(self, x, timesteps, cond_feats):
        # Generate time embeddings
        t_emb = self.time_mlp(timestep_embedding(timesteps, 320))

        # Input concat with s512
        x = torch.cat([x, cond_feats["s512"]], dim=1)
        x = self.in_conv(x)

        # Down path
        skip1, x = self.down1(x, t_emb, cond=None)
        skip2, x = self.down2(x, t_emb, cond=cond_feats["s256"])
        skip3, x = self.down3(x, t_emb, cond=cond_feats["s128"])

        # Bottleneck
        x = self.attn64(x)
        x = torch.cat([x, cond_feats["s64"]], dim=1)
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)

        # Up path
        x = self.up3(x, skip3, t_emb, cond=cond_feats["s128"])
        x = self.up2(x, skip2, t_emb, cond=cond_feats["s256"])
        x = self.up1(x, skip1, t_emb, cond=None)

        # Output prediction (x0)
        x = self.out(self.out_norm(x).clamp(-6, 6))
        return torch.tanh(x)


In [ ]:
# Decoder Trainer Class
class Cascade512Trainer:
    """
    Stage-2 training module.
    Loads precomputed latents → trains UNet512 using diffusion objective.
    Also provides visualization and composition evaluation.
    """

    def __init__(self, train_index, val_index, bs=4, lr=1e-5):
        self.accelerator = Accelerator(mixed_precision='fp16')
        self.loss_history, self.val_loss_history = [], []

        # Load datasets
        self.train_loader = DataLoader(
            PrecomputedCascadeDataset(train_index),
            batch_size=bs, shuffle=True, num_workers=4, pin_memory=True
        )
        self.val_loader = DataLoader(
            PrecomputedCascadeDataset(val_index),
            batch_size=2, shuffle=False, num_workers=2, pin_memory=True
        )

        # Model components
        self.adapter = LatentAdapter(cz=4, cond_ch=64)
        self.unet512 = UNet512(base_ch=128, cond_ch=64, time_dim=256)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.adapter.parameters()) + list(self.unet512.parameters()),
            lr=lr, betas=(0.9, 0.999), weight_decay=1e-5
        )

        # Diffusion scheduler
        self.scheduler2 = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5", subfolder="scheduler"
        )
        self.scheduler2.config.prediction_type = "sample"  # v-pred style

        # Accelerator preparation
        comps = [self.adapter, self.unet512,
                 self.train_loader, self.val_loader, self.optimizer]
        (self.adapter, self.unet512,
         self.train_loader, self.val_loader,
         self.optimizer) = self.accelerator.prepare(*comps)

        # Visualization directory
        self.vis_dir = "visualization_savepath"
        os.makedirs(self.vis_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Helper: generate pixel mask from bbox (if needed)
    # ------------------------------------------------------------------
    def _pixel_mask_from_bbox(self, bbox, img_shape):
        B, _, H, W = img_shape
        masks = torch.zeros(B, 1, H, W, device=self.accelerator.device)
        for i in range(B):
            x1, y1, x2, y2 = bbox[i]
            x1i, x2i = int(x1.item() * W), int(x2.item() * W)
            y1i, y2i = int(y1.item() * H), int(y2.item() * H)
            masks[i, :, y1i:y2i, x1i:x2i] = 1.0
        return masks

    # ------------------------------------------------------------------
    # Training/validation step
    # ------------------------------------------------------------------
    def _step(self, batch, train=True):
        masked_imgs, target_imgs, bbox, z_cond = batch

        # z_cond → device
        z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)

        # Compute multi-scale conditional features
        cond_feats = self.adapter(z_cond)

        # Sample diffusion noise
        noise = torch.randn_like(target_imgs)
        timesteps = torch.randint(
            0, self.scheduler2.config.num_train_timesteps,
            (target_imgs.shape[0],), device=target_imgs.device
        ).long()

        # Add noise
        x_noisy = self.scheduler2.add_noise(target_imgs, noise, timesteps)

        # Forward UNet
        x0_pred = self.unet512(x_noisy, timesteps, cond_feats)

        # Loss is MSE between predicted x0 and GT pixel image
        loss = F.mse_loss(x0_pred, target_imgs)

        if train:
            self.accelerator.backward(loss)

        return loss

    # ------------------------------------------------------------------
    # Validation loop
    # ------------------------------------------------------------------
    @torch.no_grad()
    def validate(self):
        self.unet512.eval()
        self.adapter.eval()

        total, n = 0.0, 0
        for batch in tqdm(self.val_loader, desc="Validating"):
            loss = self._step(batch, train=False)
            total += loss.item()
            n += 1
        return total / max(1, n)

    # ------------------------------------------------------------------
    # Visualization of samples each epoch
    # ------------------------------------------------------------------
    @torch.no_grad()
    def visualize_epoch(self, epoch_idx: int, max_batches: int = 1, steps_stage2: int = 50):
        """
        Runs the UNet in reverse diffusion mode and saves image triplets:
            masked / target / prediction
        """
        self.unet512.eval()
        self.adapter.eval()

        saved = 0
        grids = []

        self.scheduler2.set_timesteps(steps_stage2, device=self.accelerator.device)

        for batch in self.val_loader:
            masked_imgs, target_imgs, bbox, z_cond = batch

            z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
            cond_feats = self.adapter(z_cond)

            B, _, H, W = masked_imgs.shape

            # Start from noise
            x = torch.randn(
                B, 3, H, W, device=self.accelerator.device
            ) * self.scheduler2.init_noise_sigma

            for t in self.scheduler2.timesteps:
                t_batch = torch.full(
                    (B,), int(t), device=self.accelerator.device, dtype=torch.long
                )
                x0_pred = self.unet512(x, t_batch, cond_feats)
                x = self.scheduler2.step(x0_pred, t, x).prev_sample

            pred = (x.clamp(-1, 1) + 1) / 2
            masked_vis = (masked_imgs.clamp(-1, 1) + 1) / 2
            target_vis = (target_imgs.clamp(-1, 1) + 1) / 2

            triplet = torch.cat([masked_vis, target_vis, pred], dim=0)
            grid = vutils.make_grid(triplet, nrow=B, padding=2)
            grids.append(grid)
            saved += 1

            if saved >= max_batches:
                break

        if grids:
            big = torch.cat(grids, dim=1) if len(grids) > 1 else grids[0]
            out_path = os.path.join(self.vis_dir, f"epoch_{epoch_idx:03d}.png")
            vutils.save_image(big, out_path)
            self.accelerator.print(f"[Visualize] Saved {out_path}")

    # ------------------------------------------------------------------
    # Evaluation using cell composition metrics
    # ------------------------------------------------------------------
    @torch.no_grad()
    def eval_composition_batch(self, ae_model, index, steps_stage2=50, save_dir="comp_eval"):
        """
        Inference + compute cell composition distributions:
            - original GT
            - reconstructed (from masked)
            - predicted by diffusion

        Produces a 3×3 visualization grid for each sample:
            Row 1: images (recon, GT, pred)
            Row 2: composition bar charts
            Row 3: RGB histograms
        """
        self.unet512.eval()
        self.adapter.eval()
        os.makedirs(save_dir, exist_ok=True)

        self.scheduler2.set_timesteps(steps_stage2, device=self.accelerator.device)

        # Get 1 batch
        batch = next(iter(self.val_loader))
        masked_imgs, target_imgs, bbox, z_cond = batch
        B = masked_imgs.size(0)

        z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
        cond_feats = self.adapter(z_cond)

        # Reverse diffusion
        x = torch.randn(
            B, 3, masked_imgs.shape[2], masked_imgs.shape[3],
            device=self.accelerator.device
        ) * self.scheduler2.init_noise_sigma

        for t in self.scheduler2.timesteps:
            t_batch = torch.full(
                (B,), int(t), device=self.accelerator.device, dtype=torch.long
            )
            x0_pred = self.unet512(x, t_batch, cond_feats)
            x = self.scheduler2.step(x0_pred, t, x).prev_sample

        pred_imgs = (x.clamp(-1,1)+1)/2
        masked_vis = (masked_imgs.clamp(-1,1)+1)/2
        target_vis = (target_imgs.clamp(-1,1)+1)/2

        # Compute composition histograms
        fr_recon, fr_pred, fr_orig = [], [], []

        for i in range(B):
            type_recon = infer_cell_map(masked_vis[i], ae_model)
            type_pred  = infer_cell_map(pred_imgs[i], ae_model)
            type_orig  = infer_cell_map(target_vis[i], ae_model)

            dist_recon = compute_type_distribution(type_recon.squeeze(0).cpu().numpy(), num_types=25)
            dist_pred  = compute_type_distribution(type_pred.squeeze(0).cpu().numpy(), num_types=25)
            dist_orig  = compute_type_distribution(type_orig.squeeze(0).cpu().numpy(), num_types=25)

            fr_recon.append(dist_recon)
            fr_pred.append(dist_pred)
            fr_orig.append(dist_orig)

        # Save 3×3 visualization (Images / Composition Bars / RGB Histograms)
        for i in range(B):
            fig, axes = plt.subplots(3, 3, figsize=(12, 8))

            def show_img(ax, img, title):
                ax.imshow(img.permute(1, 2, 0).cpu().numpy())
                ax.set_title(title)
                ax.axis("off")

            # Row 1: images
            show_img(axes[0,0], masked_vis[i], "Recon (cond)")
            show_img(axes[0,1], target_vis[i], "Original (GT)")
            show_img(axes[0,2], pred_imgs[i],  "Predicted")

            # Row 2: composition distributions
            xs = np.arange(25)
            for ax, frac, title in zip(
                axes[1],
                [fr_recon[i], fr_orig[i], fr_pred[i]],
                ["Recon comp", "Orig comp", "Pred comp"]
            ):
                ax.bar(xs, frac, width=0.8, color='skyblue')
                ax.set_title(title)
                ax.set_ylim(0, 1)
                ax.set_xticks(xs)
                ax.set_xticklabels([f"T{i+1}" for i in xs], rotation=90, fontsize=6)
                ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3)

            # Row 3: RGB histograms
            for ax, img, title in zip(
                axes[2],
                [masked_vis[i], target_vis[i], pred_imgs[i]],
                ["Recon RGB Hist", "Orig RGB Hist", "Pred RGB Hist"]
            ):
                img_np = img.cpu().numpy()
                colors = ['r','g','b']
                for c in range(3):
                    hist, bins = np.histogram(img_np[c], bins=50, range=(0,1))
                    ax.plot(bins[:-1], hist, color=colors[c], alpha=0.7, label=colors[c].upper())
                ax.set_title(title)
                ax.legend()

            plt.tight_layout()
            out_path = os.path.join(save_dir, f"vis_comp_img{i}_{index}.png")
            plt.savefig(out_path, dpi=150)
            plt.close()
            self.accelerator.print(f"[CompEval] Saved {out_path}")

    # ------------------------------------------------------------------
    # Full training loop
    # ------------------------------------------------------------------
    def train(self, epochs=30, patience=5, vis_steps_stage2=50):
        """
        Standard training loop:
            - Training step
            - Validation
            - Visualization
            - Composition evaluation
            - Early stopping
            - Save best checkpoint
        """
        device = self.accelerator.device

        # Load autoencoder for composition inference
        ae = Autoencoder().to(device).eval()
        ae.load_state_dict(torch.load("autoencoder_path", map_location=device))

        best = float("inf")
        bad = 0

        for ep in range(1, epochs + 1):
            self.unet512.train()
            self.adapter.train()

            prog = tqdm(self.train_loader, desc=f"Epoch {ep} [Train]")
            losses = []

            for batch in prog:
                with self.accelerator.accumulate(self.unet512):
                    loss = self._step(batch, train=True)

                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.unet512.parameters(), 1.0)

                    self.optimizer.step()
                    self.optimizer.zero_grad()

                losses.append(loss.item())
                prog.set_postfix(loss=np.mean(losses))

            tr = float(np.mean(losses))
            self.loss_history.append(tr)

            va = self.validate()
            self.val_loss_history.append(va)

            self.accelerator.print(f"[Epoch {ep}] Train={tr:.4f}  Val={va:.4f}")

            # Save visualizations
            self.visualize_epoch(ep, max_batches=4, steps_stage2=vis_steps_stage2)

            # Composition evaluation
            self.eval_composition_batch(
                ae, ep, steps_stage2=vis_steps_stage2,
                save_dir="save_path"
            )

            # Early stopping logic
            if va < best - 1e-4:
                best = va
                bad = 0

                self.accelerator.wait_for_everyone()
                self.accelerator.save_state("save_path")
                self.accelerator.print(f"  >> Saved best at val {best:.4f}")

            else:
                bad += 1
                self.accelerator.print(f"  >> No improvement ({bad}/{patience})")

                if bad >= patience:
                    self.accelerator.print("Early stop.")
                    break

In [ ]:
# Entrance
train_index = "your_path/train/index.jsonl"
val_index   = "your_path/val/index.jsonl"

trainer = Cascade512Trainer(train_index, val_index, bs=4, lr=2e-5)
#trainer.accelerator.load_state("your_checkpoint_folder")
trainer.train(epochs=30, patience=10, vis_steps_stage2=200)